# Bai tap
### Thiet ke Auto Trade, chay 1p chay 1 lan - moi khi chay thi se quet du lieu Forex/ Crypto/ Chung khoan theo Timeframe 1m
### Cho ra tin hieu Buy/ Sell: MA10 > MA20 thi Buy, nguoc lai Sell

# Buoc 1: Load data

In [ ]:
def loaddataMT5(symbol, from_date, to_date, timeframe):
	import MetaTrader5 as mt5
	from datetime import datetime
	import pandas as pd 

	# Kết nối tới MetaTrader 5
	if not mt5.initialize():
		print("Khởi tạo MT5 không thành công")
		mt5.shutdown()

	from_date_str = datetime.strptime(from_date, '%Y-%m-%d')
	to_date_str = datetime.strptime(to_date, '%Y-%m-%d')
	
	# Lấy dữ liệu OHLC cho cặp tiền symbol trong khoảng thời gian đã xác định
	ohlc_data = mt5.copy_rates_range(symbol, timeframe, from_date_str, to_date_str)
	# Ngắt kết nối với MT5
	mt5.shutdown()

	# Chuyển dữ liệu nhận được thành DataFrame và hiển thị
	data = pd.DataFrame(ohlc_data)
	data['time'] = pd.to_datetime(data['time'], unit='s')

	# ohlc_df.reset_index(inplace=True)

	data = data.rename(columns={'time': 'Datetime'})        
	data = data.rename(columns={'open': 'Open'})       
	data = data.rename(columns={'high': 'High'})       
	data = data.rename(columns={'low': 'Low'})       
	data = data.rename(columns={'close': 'Close'})       
	data = data.rename(columns={'tick_volume': 'Volume'})       

	data = pd.DataFrame(data, columns=['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume'])

	return data

# Test

In [ ]:
# import MetaTrader5 as mt5
# from datetime import datetime, timedelta
# import schedule
# import time
# import pandas as pd
# import plotly.graph_objects as go
# import numpy as np

# # ##############################################Step 0: Định nghĩa các tham số##############################################
# symbol = 'US100'
# from_date = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
# to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
# timeframe = mt5.TIMEFRAME_M1

# # ##############################################Step 1: Lấy dữ liệu##############################################
# data = loaddataMT5(symbol, from_date, to_date, timeframe)

# data

# Buoc 2 Thiet ke cho chay tu dong 1m chay 1 lan => Khi chay thi goi Load data o tren => Voi data tra ve, thi tinh MA10, MA20 => Sau do tinh Buy_Signal/ Sell_Signal

In [ ]:
import MetaTrader5 as mt5
from datetime import datetime, timedelta
import schedule
import time
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import talib
import redis 

def scan_market():
	# # ##############################################Step 0: Định nghĩa các tham số##############################################
	symbol = 'US100'
	from_date = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
	to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
	timeframe = mt5.TIMEFRAME_M1

	# ##############################################Step 1: Lấy dữ liệu###########################################################
	data = loaddataMT5(symbol, from_date, to_date, timeframe)

	# ##############################################Step 2: Tinh toán chiến lược Buy_Signal, Sell_Signal##########################
	data['MA10'] = talib.SMA(data['Close'], timeperiod=10)
	data['MA20'] = talib.SMA(data['Close'], timeperiod=20)
	data['Buy_Signal'] = data['MA10'] > data['MA20']
	data['Sell_Signal'] = data['MA10'] < data['MA20']

	# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=5)
	# Tạo hash key
	hash_key = 'OG_FX_MA10, MA20'
	last_record = data.iloc[-1].copy()

	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
			last_record['Insertdate'] = datetime.now().isoformat()
		print(last_record)   
	else:
		print(last_record)   
		print('Không có tín hiệu!')

In [ ]:
# scan_market()

In [ ]:
from datetime import datetime, timedelta
import schedule
import time
import random

# Danh sách các phút cụ thể bạn muốn hàm được chạy
run_minutes = list(range(0, 60, 1)) # 0,1,2,3,4...59 : Cac phut chung ta can chay

# Thiết lập thời gian chạy cuối cùng để theo dõi
setattr(scan_market, 'last_run', None)

while True: # Luon luon kiem tra
	current_time = datetime.now() # Lay thoi gian hien tai 
	current_minute = current_time.minute # Lay phut hien
	current_second = current_time.second # Lay giay hien tai
	
	# Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
	if current_minute in run_minutes and current_second == 0: # 0 giay
		# Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
		last_run = getattr(scan_market, 'last_run', None)
		if last_run is None or last_run != current_minute:
			scan_market() # Cho chay
			# Lưu lại phút cuối cùng mà hàm đã chạy
			setattr(scan_market, 'last_run', current_minute)